# Verb Attender

Fraction of attention directed to verb positions (VB, VBD, VBG, VBN, VBP, VBZ, MD)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import torch as t
from IPython.display import Markdown, display
from shared import (
    load_model, run_and_cache, get_attention_pattern,
    show_head_pattern, show_attention_tables,
    compute_all_type_metrics, HEAD_TYPES, TYPE_ENTROPY_KEYS,
    ACTIVITY_LABELS, get_head_types, TEXT,
)

In [ ]:
model = load_model()
str_tokens, logits, cache = run_and_cache(model)

## Verb Attender — All 24 Heads

In [ ]:
tm = compute_all_type_metrics(cache, str_tokens)
ent_key = TYPE_ENTROPY_KEYS.get("verb_attention")
is_measurable = ("verb_attention", 0, 0) in tm
if is_measurable:
    results = sorted(
        [((l, h), tm[("verb_attention", l, h)]) for l in range(2) for h in range(12)],
        key=lambda x: x[1], reverse=True,
    )
    has_ent = ent_key and (ent_key, 0, 0) in tm
    if has_ent:
        lines = ["| Head | Verb Attender % | Entropy % |", "|------|-------|-------|"]  
        for (l, h), pct in results:
            ent = tm[(ent_key, l, h)]
            lines.append(f"| L{l}H{h} | {pct:.2f}% | {ent:.2f}% |")
    else:
        lines = ["| Head | Verb Attender % |", "|------|-------|"]  
        for (l, h), pct in results:
            lines.append(f"| L{l}H{h} | {pct:.2f}% |")
    display(Markdown("\n".join(lines)))
else:
    print("No programmatic metric available for this type.")